In [ ]:
import polars as pl
import duckdb

# Path ke semua file CSV di folder data/raw/
DATA_PATH = "data/raw/transjakarta_gps_*.csv"

# Membaca data secara Lazy
lazy_df = pl.scan_csv(DATA_PATH)

In [ ]:
# Menghitung total baris dan kolom
total_rows = lazy_df.select(pl.len()).collect().item()
schema = lazy_df.collect_schema()

print(f"Total Baris Data: {total_rows:,}")
print(f"Jumlah Kolom: {len(schema)}")
print("\nSkema Tipe Data:")
for col, dtype in schema.items():
    print(f"- {col}: {dtype}")

In [ ]:
# Mengambil sampel 5 baris pertama
df_head = lazy_df.limit(5).collect()
print(df_head)

In [ ]:
# Pengecekan Missing Values per kolom
missing_counts = lazy_df.select([
    pl.col(c).null_count().alias(c) for c in schema.keys()
]).collect()

print("Jumlah Nilai Kosong (Null):")
print(missing_counts)

In [ ]:
# Profiling statistik deskriptif dengan DuckDB SUMMARIZE
con = duckdb.connect()
summary_df = con.execute(f"SUMMARIZE SELECT * FROM read_csv_auto('{DATA_PATH}')").df()

print(summary_df[['column_name', 'column_type', 'min', 'max', 'approx_unique', 'null_percentage']])